# Pratica 12 — Progetto integratore e ponte verso i transformer

**Corso:** NLP Classico con Python · **Lezione 12** · **Tempo stimato:** 90-110 minuti
**Notebook di lezione:** `notebooks_clase/clase12_proyecto_puente_transformers_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Questa pratica usa le **predizioni precalcolate** del *transformer* (`datos/plan_b_transformer_resenas.json`), non scarica il modello: così puoi farla senza connessione e in pochi secondi. Se vuoi eseguire il modello reale, fallo nel notebook di lezione.
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas` e `scikit-learn`.

---
## Cella 0 — Setup e dati

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

IDIOMAS = ("es", "it")   # dato-es
RESENAS = {idioma: pd.read_csv(cargar_datos(f"resenas_{idioma}.csv")) for idioma in IDIOMAS}
with open(cargar_datos("plan_b_transformer_resenas.json"), encoding="utf-8") as f:
    PLAN_B = json.load(f)
ETIQUETAS = ("negativo", "neutro", "positivo")   # dato-es

print({idioma: df.shape for idioma, df in RESENAS.items()})
print("Librerie e dati caricati.")

## Esercizio 1 — La `Pipeline` classica, di nuovo

Completa `crear_pipeline_clasico()`: una `Pipeline` con due passi, `"tfidf"` (`TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)`, le stesse impostazioni delle lezioni 5 e 6) e `"clf"` (`LogisticRegression(max_iter=1000)`).

In [ ]:
def crear_pipeline_clasico():
    return None      # TODO: Pipeline([("tfidf", ...), ("clf", ...)])

# Controllo automatico
pipe = crear_pipeline_clasico()
assert pipe is not None, "Completa le righe contrassegnate con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline deve avere i passi «tfidf» e «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2) and pipe.named_steps["tfidf"].sublinear_tf is True
assert isinstance(pipe.named_steps["clf"], LogisticRegression) and pipe.named_steps["clf"].max_iter == 1000
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline si restituisce senza adattarla"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Addestrare e valutare, con la partizione del capitolo

Completa `entrenar_y_evaluar(idioma)`: divide `RESENAS[idioma]` in 70% addestramento / 30% test (`train_test_split(..., test_size=0.3, stratify=df["etiqueta"], random_state=42)`), addestra `crear_pipeline_clasico()` con l'addestramento e restituisce `(accuratezza, f1_macro, prueba)` sul test, dove `prueba` è il `DataFrame` di test **con l'indice reimpostato** (`reset_index(drop=True)`).

In [ ]:
def entrenar_y_evaluar(idioma):
    df = RESENAS[idioma]
    ent, prueba = None, None    # TODO: train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    exactitud, f1_macro = None, None    # TODO: addestra la Pipeline con ent e valuta su prueba
    prueba_final = prueba.reset_index(drop=True) if prueba is not None else None
    return exactitud, f1_macro, prueba_final

# Controllo automatico
REFERENCIA = {"es": (0.722, 0.722), "it": (0.789, 0.786)}   # dato-es
for idioma in IDIOMAS:
    resultado = entrenar_y_evaluar(idioma)
    assert resultado is not None and resultado[0] is not None, "Completa le righe contrassegnate con TODO"
    exactitud, f1_macro, prueba = resultado
    assert len(prueba) == 90, "il 30% di 300 recensioni sono 90"
    ref_acc, ref_f1 = REFERENCIA[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "con questa partizione e seme, l'accuratezza deve coincidere con quella del capitolo"
    assert abs(f1_macro - ref_f1) < 0.01, "lo stesso per l'F1 macro"
    print(idioma, "accuratezza:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Esercizio 2 corretto ✔")

## Esercizio 3 — La regola di mappatura delle stelle

Completa `mapear_estrellas(etiqueta_estrellas)`: riceve una stringa come `"1 star"` o `"4 stars"` e restituisce `"negativo"` (1-2), `"neutro"` (3) o `"positivo"` (4-5), secondo la regola giustificata nel capitolo (sezione 3.2). Suggerimento: il primo carattere della stringa è il numero.

In [ ]:
def mapear_estrellas(etiqueta_estrellas):
    return None      # TODO: 1-2 -> "negativo", 3 -> "neutro", 4-5 -> "positivo"

# Controllo automatico
assert mapear_estrellas("1 star") is not None, "Completa le righe contrassegnate con TODO"
assert mapear_estrellas("1 star") == "negativo" and mapear_estrellas("2 stars") == "negativo"
assert mapear_estrellas("3 stars") == "neutro"
assert mapear_estrellas("4 stars") == "positivo" and mapear_estrellas("5 stars") == "positivo"
assert [mapear_estrellas(f"{n} stars") for n in range(1, 6)] == ["negativo", "negativo", "neutro", "positivo", "positivo"]
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Valutare il transformer con il piano B, sullo stesso test

Completa `evaluar_transformer(idioma, prueba)`: usa `PLAN_B["idiomas"][idioma]` per costruire, **nello stesso ordine di `prueba["id"]`**, la lista delle stelle precalcolate (`estrellas`) e la loro traduzione con `mapear_estrellas`; restituisce `(exactitud, f1_macro)` confrontando quelle predizioni con `prueba["etiqueta"]`. `prueba` è il `DataFrame` di test restituito da `entrenar_y_evaluar` (esercizio 2): così classico e *transformer* si confrontano su esattamente gli stessi testi.

In [ ]:
def evaluar_transformer(idioma, prueba):
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = None    # TODO: dict(zip(datos["id"], datos["estrellas"])), con le chiavi come testo
    estrellas = None           # TODO: una stella per riga di prueba, nello stesso ordine
    pred = None                # TODO: mapear_estrellas applicato a ogni stella
    return None, None          # TODO: (accuratezza, f1_macro)

# Controllo automatico
REFERENCIA_TRAFO = {"es": (0.856, 0.852), "it": (0.844, 0.842)}   # dato-es
for idioma in IDIOMAS:
    _, _, prueba = entrenar_y_evaluar(idioma)
    resultado = evaluar_transformer(idioma, prueba)
    assert resultado is not None and resultado[0] is not None, "Completa le righe contrassegnate con TODO"
    exactitud, f1_macro = resultado
    ref_acc, ref_f1 = REFERENCIA_TRAFO[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "deve coincidere con le predizioni precalcolate del piano B, allineate per id"
    assert abs(f1_macro - ref_f1) < 0.01
    print(idioma, "transformer accuratezza:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Leggere i coefficienti del classico

Completa `top_palabras(pipe, clase, k)`: data una `Pipeline` già adattata (TF-IDF + `LogisticRegression`), restituisce l'elenco delle `k` parole con il coefficiente maggiore per `clase` (dalla maggiore alla minore), usando `pipe.named_steps["clf"].classes_` per trovare la riga corretta.

In [ ]:
def top_palabras(pipe, clase, k=5):
    vocab = pipe.named_steps["tfidf"].get_feature_names_out()
    fila = None    # TODO: indice di "clase" in pipe.named_steps["clf"].classes_
    coef = pipe.named_steps["clf"].coef_[fila]
    return None    # TODO: le k parole con il coefficiente maggiore, dalla maggiore alla minore (np.argsort)

# Controllo automatico
IDIOMA_EJ = IDIOMAS[0]   # dato-es
ent_ej, _ = train_test_split(RESENAS[IDIOMA_EJ], test_size=0.3, stratify=RESENAS[IDIOMA_EJ]["etiqueta"], random_state=42)
pipe_ej = crear_pipeline_clasico().fit(ent_ej["texto"], ent_ej["etiqueta"])
top = top_palabras(pipe_ej, "positivo", k=5)
assert top is not None, "Completa le righe contrassegnate con TODO"
assert len(top) == 5, "5 parole"
vocab_ej = pipe_ej.named_steps["tfidf"].get_feature_names_out()
coef_pos = pipe_ej.named_steps["clf"].coef_[list(pipe_ej.named_steps["clf"].classes_).index("positivo")]
valores = [coef_pos[list(vocab_ej).index(p)] for p in top]
assert valores == sorted(valores, reverse=True), "ordina dal coefficiente maggiore al minore"
assert top[0] == vocab_ej[int(np.argmax(coef_pos))], "la prima parola deve avere il coefficiente più alto"
print("top positivo, lingua di esempio:", IDIOMA_EJ, top)
print("Esercizio 5 corretto ✔")

---
## Sfida finale — ciascun metodo fallisce su una classe diversa

Integra quanto appreso. Completa `clase_mas_dificil(idioma)`, che deve restituire una tupla `(clase_clasico, clase_transformer)` con il nome della classe (`"positivo"`, `"negativo"` o `"neutro"`) con il **richiamo più basso di ciascun metodo separatamente** (non mediati: la sfida di questa lezione è scoprire che **non è la stessa classe**):

- Addestra il classico con `crear_pipeline_clasico()` sulla stessa partizione 70/30 (`train_test_split(..., test_size=0.3, stratify=df["etiqueta"], random_state=42)`) e predici sul test,
- Calcola le predizioni del *transformer* con la stessa logica dell'esercizio 4,
- Con `precision_recall_fscore_support(prueba["etiqueta"], predicciones, labels=ETIQUETAS)`, ottieni il *recall* di ogni classe per ogni metodo e cerca, in ciascuno separatamente, la classe con il richiamo più basso (`ETIQUETAS[np.argmin(...)]`).

Verifica che il classico e il *transformer* falliscano su classi **diverse**, e che il pattern si ripeta in entrambe le lingue.

In [ ]:
def clase_mas_dificil(idioma):
    df = RESENAS[idioma]
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    prueba = prueba.reset_index(drop=True)
    pipe = crear_pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred_clasico = None      # TODO: pipe.predict(prueba["texto"])
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = dict(zip([str(x) for x in datos["id"]], datos["estrellas"]))
    pred_trafo = None        # TODO: mapear_estrellas della stella di ogni id di prueba, nel suo ordine
    recall_clasico = None    # TODO: precision_recall_fscore_support(prueba["etiqueta"], pred_clasico, labels=ETIQUETAS)[1]
    recall_trafo = None      # TODO: lo stesso con pred_trafo
    if recall_clasico is None or recall_trafo is None:
        return None
    return ETIQUETAS[int(np.argmin(recall_clasico))], ETIQUETAS[int(np.argmin(recall_trafo))]

for idioma in IDIOMAS:
    resultado = clase_mas_dificil(idioma)
    assert resultado is not None, "Completa le righe contrassegnate con TODO"
    clase_clasico, clase_trafo = resultado
    assert clase_clasico is not None and clase_trafo is not None, "Completa le righe contrassegnate con TODO"
    assert clase_clasico == "positivo", "secondo il capitolo (sezione 4.1), il classico fallisce di più su «positivo» in entrambe le lingue"
    assert clase_trafo == "neutro", "secondo il capitolo (sezione 4.1), il transformer fallisce di più su «neutro» in entrambe le lingue"
    assert clase_clasico != clase_trafo, "il punto della sfida: ciascun metodo fallisce su una classe diversa"
    print(idioma, "| classico fallisce di più su:", clase_clasico, "| transformer fallisce di più su:", clase_trafo)
print("Sfida finale corretta ✔")

**Le tue conclusioni (scrivi 3-4 frasi):** il classico fallisce di più su `positivo` (poco vocabolario da 210 esempi) e il *transformer* di più su `neutro` (giudizi misti, difficili anche per un modello grande). Cosa ti dice questo sul combinare i due metodi in un progetto reale? Se dovessi migliorare solo una classe con più dati, a quale lingua o metodo dedicheresti lo sforzo? Cosa diresti a qualcuno che conclude "il transformer vince sempre" a partire da questa lezione?

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def crear_pipeline_clasico():
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)), ("clf", LogisticRegression(max_iter=1000))])

# Controllo automatico
pipe = crear_pipeline_clasico()
assert pipe is not None, "Completa le righe contrassegnate con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline deve avere i passi «tfidf» e «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2) and pipe.named_steps["tfidf"].sublinear_tf is True
assert isinstance(pipe.named_steps["clf"], LogisticRegression) and pipe.named_steps["clf"].max_iter == 1000
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline si restituisce senza adattarla"
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def entrenar_y_evaluar(idioma):
    df = RESENAS[idioma]
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    pipe = crear_pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred = pipe.predict(prueba["texto"])
    exactitud = accuracy_score(prueba["etiqueta"], pred)
    f1_macro = f1_score(prueba["etiqueta"], pred, average="macro")
    return exactitud, f1_macro, prueba.reset_index(drop=True)

# Controllo automatico
REFERENCIA = {"es": (0.722, 0.722), "it": (0.789, 0.786)}   # dato-es
for idioma in IDIOMAS:
    resultado = entrenar_y_evaluar(idioma)
    assert resultado is not None and resultado[0] is not None, "Completa le righe contrassegnate con TODO"
    exactitud, f1_macro, prueba = resultado
    assert len(prueba) == 90, "il 30% di 300 recensioni sono 90"
    ref_acc, ref_f1 = REFERENCIA[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "con questa partizione e seme, l'accuratezza deve coincidere con quella del capitolo"
    assert abs(f1_macro - ref_f1) < 0.01, "lo stesso per l'F1 macro"
    print(idioma, "accuratezza:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
def mapear_estrellas(etiqueta_estrellas):
    n = int(etiqueta_estrellas[0])
    if n <= 2:
        return "negativo"
    if n == 3:
        return "neutro"
    return "positivo"

# Controllo automatico
assert mapear_estrellas("1 star") is not None, "Completa le righe contrassegnate con TODO"
assert mapear_estrellas("1 star") == "negativo" and mapear_estrellas("2 stars") == "negativo"
assert mapear_estrellas("3 stars") == "neutro"
assert mapear_estrellas("4 stars") == "positivo" and mapear_estrellas("5 stars") == "positivo"
assert [mapear_estrellas(f"{n} stars") for n in range(1, 6)] == ["negativo", "negativo", "neutro", "positivo", "positivo"]
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def evaluar_transformer(idioma, prueba):
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = dict(zip([str(x) for x in datos["id"]], datos["estrellas"]))
    estrellas = [estrellas_por_id[str(i)] for i in prueba["id"]]
    pred = [mapear_estrellas(e) for e in estrellas]
    exactitud = accuracy_score(prueba["etiqueta"], pred)
    f1_macro = f1_score(prueba["etiqueta"], pred, average="macro")
    return exactitud, f1_macro

# Controllo automatico
REFERENCIA_TRAFO = {"es": (0.856, 0.852), "it": (0.844, 0.842)}   # dato-es
for idioma in IDIOMAS:
    _, _, prueba = entrenar_y_evaluar(idioma)
    resultado = evaluar_transformer(idioma, prueba)
    assert resultado is not None and resultado[0] is not None, "Completa le righe contrassegnate con TODO"
    exactitud, f1_macro = resultado
    ref_acc, ref_f1 = REFERENCIA_TRAFO[idioma]
    assert abs(exactitud - ref_acc) < 0.01, "deve coincidere con le predizioni precalcolate del piano B, allineate per id"
    assert abs(f1_macro - ref_f1) < 0.01
    print(idioma, "transformer accuratezza:", round(exactitud, 3), "| F1 macro:", round(f1_macro, 3))
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
def top_palabras(pipe, clase, k=5):
    vocab = pipe.named_steps["tfidf"].get_feature_names_out()
    fila = list(pipe.named_steps["clf"].classes_).index(clase)
    coef = pipe.named_steps["clf"].coef_[fila]
    return [vocab[i] for i in np.argsort(-coef)[:k]]

# Controllo automatico
IDIOMA_EJ = IDIOMAS[0]   # dato-es
ent_ej, _ = train_test_split(RESENAS[IDIOMA_EJ], test_size=0.3, stratify=RESENAS[IDIOMA_EJ]["etiqueta"], random_state=42)
pipe_ej = crear_pipeline_clasico().fit(ent_ej["texto"], ent_ej["etiqueta"])
top = top_palabras(pipe_ej, "positivo", k=5)
assert top is not None, "Completa le righe contrassegnate con TODO"
assert len(top) == 5, "5 parole"
vocab_ej = pipe_ej.named_steps["tfidf"].get_feature_names_out()
coef_pos = pipe_ej.named_steps["clf"].coef_[list(pipe_ej.named_steps["clf"].classes_).index("positivo")]
valores = [coef_pos[list(vocab_ej).index(p)] for p in top]
assert valores == sorted(valores, reverse=True), "ordina dal coefficiente maggiore al minore"
assert top[0] == vocab_ej[int(np.argmax(coef_pos))], "la prima parola deve avere il coefficiente più alto"
print("top positivo, lingua di esempio:", IDIOMA_EJ, top)
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def clase_mas_dificil(idioma):
    df = RESENAS[idioma]
    ent, prueba = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    prueba = prueba.reset_index(drop=True)
    pipe = crear_pipeline_clasico().fit(ent["texto"], ent["etiqueta"])
    pred_clasico = pipe.predict(prueba["texto"])
    datos = PLAN_B["idiomas"][idioma]
    estrellas_por_id = dict(zip([str(x) for x in datos["id"]], datos["estrellas"]))
    pred_trafo = [mapear_estrellas(estrellas_por_id[str(i)]) for i in prueba["id"]]
    recall_clasico = precision_recall_fscore_support(prueba["etiqueta"], pred_clasico, labels=ETIQUETAS)[1]
    recall_trafo = precision_recall_fscore_support(prueba["etiqueta"], pred_trafo, labels=ETIQUETAS)[1]
    return ETIQUETAS[int(np.argmin(recall_clasico))], ETIQUETAS[int(np.argmin(recall_trafo))]

for idioma in IDIOMAS:
    resultado = clase_mas_dificil(idioma)
    assert resultado is not None, "Completa le righe contrassegnate con TODO"
    clase_clasico, clase_trafo = resultado
    assert clase_clasico is not None and clase_trafo is not None, "Completa le righe contrassegnate con TODO"
    assert clase_clasico == "positivo", "secondo il capitolo (sezione 4.1), il classico fallisce di più su «positivo» in entrambe le lingue"
    assert clase_trafo == "neutro", "secondo il capitolo (sezione 4.1), il transformer fallisce di più su «neutro» in entrambe le lingue"
    assert clase_clasico != clase_trafo, "il punto della sfida: ciascun metodo fallisce su una classe diversa"
    print(idioma, "| classico fallisce di più su:", clase_clasico, "| transformer fallisce di più su:", clase_trafo)
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (zero-shot, regola di mappatura, costo computazionale, spiegabilità, attenzione)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*